# NWJ Discriminator MI Estimator (v2)

Same pair-transformer critic architecture as `discriminator.ipynb`, with an improved **training protocol**:

- more training negatives and batches (closer to audit),
- symmetric NWJ exp clamp,
- ramped hard / in-batch negatives,
- validation checkpoint gates (reject saturated or unstable critics).

Audit protocol and row format are unchanged for direct comparison with v1.


## 0. Kaggle / remote setup

On Kaggle, this cell clones the project from GitHub and adds `notebooks/` to `sys.path`. Locally, it reuses the checkout next to this notebook.

Optional environment variables:
- `INVERSE_POSITIONING_REPO` (default: `https://github.com/romanbranovets/inverse_positioning.git`)
- `INVERSE_POSITIONING_REF` (default: `main`)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO = os.environ.get(
    "INVERSE_POSITIONING_REPO",
    "https://github.com/romanbranovets/inverse_positioning.git",
)
GITHUB_REF = os.environ.get("INVERSE_POSITIONING_REF", "main")

NOTEBOOK_ROOT = Path.cwd()
LOCAL_CANDIDATES = [
    NOTEBOOK_ROOT,
    NOTEBOOK_ROOT / "notebooks",
    NOTEBOOK_ROOT.parent,
    NOTEBOOK_ROOT.parent / "notebooks",
]


def find_shared_root():
    for root in LOCAL_CANDIDATES:
        if (root / "shared" / "trajectory_sampler.py").exists():
            return root
    return None


shared_root = find_shared_root()
if shared_root is None:
    clone_dir = NOTEBOOK_ROOT / "_inverse_positioning"
    if clone_dir.exists():
        import shutil

        shutil.rmtree(clone_dir)
    print(f"Cloning {GITHUB_REPO} @ {GITHUB_REF} ...")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GITHUB_REF,
            GITHUB_REPO,
            str(clone_dir),
        ],
        check=True,
    )
    shared_root = clone_dir / "notebooks"
    if not (shared_root / "shared" / "trajectory_sampler.py").exists():
        raise FileNotFoundError(
            "Cloned repo missing notebooks/shared. Push latest repo and set INVERSE_POSITIONING_REF."
        )

if str(shared_root) not in sys.path:
    sys.path.insert(0, str(shared_root))

try:
    import plotly  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)

print(f"shared_root={shared_root}")


## 1. Setup

The synthetic field and trajectory generator are imported from the shared modules. Training samples, validation samples, and final audit samples are generated independently on the fly.


In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.nn import functional as F

from shared.sphere_utils import *
from shared.magentic_field import *
from shared.trajectory_sampler import *

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device={DEVICE}, step={STEP_METERS:.1f} m, angular step={STEP_RAD:.3e} rad")

FEATURE_MEAN, FEATURE_STD = estimate_feature_normalization(device=DEVICE)


D_MODEL = 256
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.05
TRAIN_EPOCHS = 90
TRAIN_BATCHES_PER_EPOCH = 8
TRAIN_BATCH_SIZE = 512
LEARNING_RATE = 2e-4
NWJ_LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
PLOT_EVERY = 2
BCE_WARMUP_EPOCHS = 6
NWJ_BCE_WEIGHT = 0.05
TRAIN_NWJ_EXP_CLAMP = 6.0
LOGIT_ABS_CLAMP = 22.0
TARGET_AUDIT_BITS = 30.0
CONFIDENCE_LOGIT_ABS_CLAMP = 22.0
NEGATIVES_PER_POSITIVE = 32

VALIDATION_EVERY = 2
VALIDATION_BATCH_SIZE = 512
VALIDATION_BATCHES_PER_POINT = 2
VALIDATION_NEGATIVES_PER_POSITIVE = 32
EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
EVAL_BATCH_SIZE = 512
EVAL_BATCHES_PER_POINT = 32
EVAL_NEGATIVES_PER_POSITIVE = 128
EVAL_LOGIT_CHUNK_SIZE = 1024
EVAL_TOTAL_CONFIDENCE_DELTA = 0.01

NWJ_EARLY_STOP_MIN_EPOCHS = 24
NWJ_EARLY_STOP_PATIENCE = 10
NWJ_EARLY_STOP_MIN_DELTA_BITS = 0.05

IN_BATCH_NEGATIVES = 4
HARD_NEGATIVE_FRACTION = 0.5
HARD_NEGATIVE_RAMP_EPOCHS = 15
IN_BATCH_WARMUP_EPOCHS = 10
VAL_CHECKPOINT_MIN_EPOCH = 20
VAL_CHECKPOINT_MIN_NWJ_BITS = 0.0
VAL_CHECKPOINT_MAX_NEG_PENALTY_BITS = 1.5
VAL_CHECKPOINT_MAX_POS_BITS = 20.0
VAL_AUDIT_NEGATIVES_EVERY = 8

RESIDUAL_CLIP_SIGMA = 30.0
ENDPOINT_OFFSET_SCALE_M = 2_000_000.0
PAIR_FEATURE_DIM = FEATURE_DIM + 3 + 1 + 3 + 2

## 2. Critic Model

The critic scores an `(endpoint, trajectory)` pair. Positive pairs use the true endpoint of the generated trajectory; product-negative pairs reuse the trajectory with independently sampled uniform endpoints. BCE is used only as a short warmup. After warmup, training uses a stabilized NWJ-style objective. That training surrogate may cap the exponential penalty for numerical stability, but it is not used as the reported MI estimate.

The final audit uses the strict NWJ formula for a clipped critic with a clamp large enough to represent 30 bits. The audit confidence interval is computed from independent batch-level NWJ estimates; the old worst-case range certificate is intentionally not used for 30-bit mode because its radius scales with `exp(clamp - 1)` and becomes computationally useless.


Architecture identical to v1; v2 changes are in sampling, loss stabilization, and checkpoint selection.


In [ ]:
def log_map_sphere_2d_meters(base, target):
    dot = (target * base).sum(dim=-1).clamp(-1.0, 1.0)
    tangent = target - dot[:, None] * base
    sin_theta = tangent.norm(dim=-1).clamp_min(1e-12)
    theta = torch.atan2(sin_theta, dot)
    direction = tangent / sin_theta[:, None]
    e1, e2 = tangent_basis(base)
    v_m = EARTH_RADIUS_M * theta[:, None] * direction
    return torch.stack([(v_m * e1).sum(dim=-1), (v_m * e2).sum(dim=-1)], dim=-1)


@torch.no_grad()
def reconstruct_candidate_path(endpoint, x, pad_mask):
    batch_size, token_count, _ = x.shape
    positions_rev = []
    u = endpoint
    positions_rev.append(u)
    for t in range(token_count - 2, -1, -1):
        du = x[:, t, 6:8]
        e1, e2 = tangent_basis(u)
        back_step = -du[:, 0:1] * e1 - du[:, 1:2] * e2
        u = exp_map_sphere(u, back_step)
        positions_rev.append(u)
    return torch.stack(list(reversed(positions_rev)), dim=1)


def make_pair_features(x, pad_mask, endpoint):
    candidate_path = reconstruct_candidate_path(endpoint, x, pad_mask)
    predicted_B = B(candidate_path.reshape(-1, 3)).reshape(candidate_path.shape)
    residual_sigma = (x[:, :, :3] - predicted_B) / EPS_B_STD
    residual_B = torch.clamp(residual_sigma / RESIDUAL_CLIP_SIGMA, -1.0, 1.0)
    residual_energy = torch.log1p(residual_sigma.square().sum(dim=-1, keepdim=True))

    last_idx = (~pad_mask).sum(dim=1).sub(1).clamp_min(0)
    final_B = x[torch.arange(x.shape[0], device=x.device), last_idx, :3]
    anchor = normalize(final_B)
    endpoint_offset = (
        log_map_sphere_2d_meters(anchor, endpoint) / ENDPOINT_OFFSET_SCALE_M
    )
    endpoint_B = B(endpoint)

    endpoint_offset_tokens = endpoint_offset[:, None, :].expand(-1, x.shape[1], -1)
    endpoint_B_tokens = endpoint_B[:, None, :].expand(-1, x.shape[1], -1)
    scaled_x = (x - FEATURE_MEAN) / FEATURE_STD
    return torch.cat(
        [
            scaled_x,
            residual_B,
            residual_energy,
            endpoint_B_tokens,
            endpoint_offset_tokens,
        ],
        dim=-1,
    )


class EndpointTrajectoryDiscriminator(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(PAIR_FEATURE_DIM, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
        )
        self.position_embedding = nn.Parameter(torch.zeros(1, MAX_STEPS + 1, d_model))
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.attention = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x, pad_mask, endpoint):
        pair_features = make_pair_features(x, pad_mask, endpoint)
        h = self.input_projection(pair_features)
        h = h + self.position_embedding[:, : h.shape[1], :]
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        attention_logits = (
            self.attention(h).squeeze(-1).masked_fill(pad_mask, -torch.inf)
        )
        weights = torch.softmax(attention_logits, dim=-1)
        pooled = torch.sum(h * weights[:, :, None], dim=1)
        return self.head(pooled).squeeze(-1)




def trajectory_anchor(x, pad_mask):
    last_idx = (~pad_mask).sum(dim=1).sub(1).clamp_min(0)
    final_b = x[torch.arange(x.shape[0], device=x.device), last_idx, :3]
    return normalize(final_b)


def slerp_sphere(base, target, t):
    base = normalize(base)
    target = normalize(target)
    dot = (base * target).sum(dim=-1, keepdim=True).clamp(-1.0 + 1e-6, 1.0 - 1e-6)
    omega = torch.acos(dot)
    sin_omega = torch.sin(omega).clamp_min(1e-6)
    t = t[..., None] if t.dim() == 1 else t
    weight_base = torch.sin((1.0 - t) * omega) / sin_omega
    weight_target = torch.sin(t * omega) / sin_omega
    return normalize(weight_base * base + weight_target * target)


def perturb_on_sphere(base, std_rad):
    base = normalize(base)
    e1, e2 = tangent_basis(base)
    coeffs = torch.randn(*base.shape[:-1], 2, device=base.device, dtype=base.dtype)
    tangent_m = std_rad * EARTH_RADIUS_M * (
        coeffs[..., 0:1] * e1 + coeffs[..., 1:2] * e2
    )
    return exp_map_sphere(base, tangent_m)


def sample_hard_endpoints(anchor, true_endpoint, count, device, dtype):
    if count <= 0:
        return torch.empty(0, 3, device=device, dtype=dtype)
    anchor = normalize(anchor.reshape(-1, 3)[0]).expand(count, -1)
    true_endpoint = normalize(true_endpoint.reshape(-1, 3)[0]).expand(count, -1)
    n_near = max(1, count // 3)
    n_slerp = max(1, count // 3)
    n_true = max(1, count - n_near - n_slerp)
    near = perturb_on_sphere(anchor[:n_near], std_rad=0.08)
    t = 0.35 + 0.55 * torch.rand(n_slerp, device=device, dtype=dtype)
    mid = slerp_sphere(anchor[:n_slerp], true_endpoint[:n_slerp], t)
    near_true = perturb_on_sphere(true_endpoint[:n_true], std_rad=0.05)
    points = torch.cat([near, mid, near_true], dim=0)
    if points.shape[0] < count:
        extra = sample_uniform_sphere(count - points.shape[0], device=device).to(dtype)
        points = torch.cat([points, extra], dim=0)
    return normalize(points[:count])


def hard_negative_fraction_for_epoch(epoch, warmup_epochs, ramp_epochs, target_fraction):
    if epoch <= warmup_epochs:
        return 0.0
    if ramp_epochs <= 0:
        return target_fraction
    progress = (epoch - warmup_epochs) / float(ramp_epochs)
    return target_fraction * min(1.0, max(0.0, progress))


def sample_mixed_endpoints(
    x,
    pad_mask,
    true_endpoint,
    negative_count,
    hard_fraction=0.0,
    in_batch_count=0,
):
    batch_size = true_endpoint.shape[0]
    device = true_endpoint.device
    dtype = true_endpoint.dtype
    anchor = trajectory_anchor(x, pad_mask)
    hard_count = int(round(negative_count * hard_fraction))
    uniform_count = negative_count - hard_count
    pieces = []
    if uniform_count > 0:
        uniform = sample_uniform_sphere(batch_size * uniform_count, device=device).to(dtype)
        pieces.append(uniform.reshape(batch_size, uniform_count, 3))
    if hard_count > 0:
        hard = []
        for row in range(batch_size):
            hard.append(
                sample_hard_endpoints(
                    anchor[row : row + 1],
                    true_endpoint[row : row + 1],
                    hard_count,
                    device,
                    dtype,
                )
            )
        pieces.append(torch.stack(hard, dim=0))
    if in_batch_count > 0:
        perm = torch.randperm(batch_size, device=device)
        shifted = true_endpoint[perm]
        same = (shifted - true_endpoint).norm(dim=-1) < 1e-5
        if same.any():
            shifted = torch.where(
                same[:, None],
                perturb_on_sphere(true_endpoint, std_rad=0.2),
                shifted,
            )
        pieces.append(
            shifted[:, None, :].expand(-1, in_batch_count, -1).reshape(
                batch_size, in_batch_count, 3
            )
        )
    return normalize(torch.cat(pieces, dim=1))



def sample_product_endpoints(batch_size, negative_count, device=DEVICE):
    return sample_uniform_sphere(batch_size * negative_count, device=device).reshape(
        batch_size, negative_count, 3
    )


def contrastive_logits(
    model,
    x,
    pad_mask,
    endpoint,
    negative_count=NEGATIVES_PER_POSITIVE,
    negative_chunk_size=None,
    hard_negative_fraction=0.0,
    in_batch_negatives=0,
):
    positive_logit = model(x, pad_mask, endpoint)
    if hard_negative_fraction <= 0.0 and in_batch_negatives <= 0:
        negative_endpoint = sample_product_endpoints(
            endpoint.shape[0], negative_count, endpoint.device
        )
    else:
        negative_endpoint = sample_mixed_endpoints(
            x,
            pad_mask,
            endpoint,
            negative_count,
            hard_fraction=hard_negative_fraction,
            in_batch_count=in_batch_negatives,
        )
    flat_negative_endpoint = negative_endpoint.reshape(-1, 3)
    if negative_chunk_size is None:
        repeated_x = x.repeat_interleave(negative_count, dim=0)
        repeated_mask = pad_mask.repeat_interleave(negative_count, dim=0)
        negative_logit = model(repeated_x, repeated_mask, flat_negative_endpoint)
    else:
        source_index = torch.arange(x.shape[0], device=x.device).repeat_interleave(
            negative_count
        )
        chunks = []
        for start in range(0, flat_negative_endpoint.shape[0], negative_chunk_size):
            stop = min(start + negative_chunk_size, flat_negative_endpoint.shape[0])
            row_index = source_index[start:stop]
            chunks.append(
                model(
                    x.index_select(0, row_index),
                    pad_mask.index_select(0, row_index),
                    flat_negative_endpoint[start:stop],
                )
            )
        negative_logit = torch.cat(chunks, dim=0)
    return positive_logit, negative_logit


def make_discriminator_batch(batch_size, step_count, negative_total=NEGATIVES_PER_POSITIVE + IN_BATCH_NEGATIVES):
    x, pad_mask, endpoint, _ = make_trajectories(batch_size, step_counts=step_count)
    labels = torch.cat(
        [
            torch.ones(batch_size, device=DEVICE),
            torch.zeros(batch_size * negative_total, device=DEVICE),
        ],
        dim=0,
    )
    return x, pad_mask, endpoint, labels


def bounded_logits(logits, clamp_abs=LOGIT_ABS_CLAMP):
    return logits.clamp(-clamp_abs, clamp_abs)


def nwj_terms(positive_logits, negative_logits, clamp_abs=LOGIT_ABS_CLAMP):
    positive_t = bounded_logits(positive_logits, clamp_abs)
    negative_t = bounded_logits(negative_logits, clamp_abs)
    negative_exp = torch.exp(negative_t - 1.0)
    return positive_t, negative_exp


def discriminator_nwj_nats(positive_logits, negative_logits, clamp_abs=LOGIT_ABS_CLAMP):
    positive_t, negative_exp = nwj_terms(positive_logits, negative_logits, clamp_abs)
    return positive_t.mean() - negative_exp.mean()


def discriminator_nwj_training_nats(positive_logits, negative_logits):
    positive_t = bounded_logits(positive_logits, LOGIT_ABS_CLAMP)
    negative_t = bounded_logits(negative_logits, LOGIT_ABS_CLAMP)
    negative_exp = torch.exp(
        (negative_t - 1.0).clamp(min=-TRAIN_NWJ_EXP_CLAMP, max=TRAIN_NWJ_EXP_CLAMP)
    )
    return positive_t.mean() - negative_exp.mean()


def discriminator_nwj_training_loss(positive_logits, negative_logits):
    return -discriminator_nwj_training_nats(positive_logits, negative_logits)


@torch.no_grad()
def discriminator_nwj_training_bits(positive_logits, negative_logits):
    return float(
        (
            discriminator_nwj_training_nats(positive_logits, negative_logits)
            / math.log(2.0)
        ).cpu()
    )


@torch.no_grad()
def discriminator_nwj_bits(positive_logits, negative_logits, clamp_abs=LOGIT_ABS_CLAMP):
    return float(
        (
            discriminator_nwj_nats(positive_logits, negative_logits, clamp_abs)
            / math.log(2.0)
        ).cpu()
    )


@torch.no_grad()
def normal_quantile(probability):
    p = torch.tensor(float(probability), dtype=torch.float64)
    return float(torch.distributions.Normal(0.0, 1.0).icdf(p).item())


@torch.no_grad()
def nwj_summary_from_logits(positive_logits, negative_logits, clamp_abs):
    positive_t, negative_exp = nwj_terms(positive_logits, negative_logits, clamp_abs)
    positive_mean = positive_t.mean().double()
    negative_mean = negative_exp.mean().double()
    nwj_nats = positive_mean - negative_mean
    return dict(
        nwj_nats=float(nwj_nats.cpu()),
        positive_mean_nats=float(positive_mean.cpu()),
        negative_penalty_nats=float(negative_mean.cpu()),
        positive_samples=int(positive_t.numel()),
        negative_samples=int(negative_exp.numel()),
    )


@torch.no_grad()
def batch_mean_lcb(values, delta):
    values = torch.as_tensor(values, dtype=torch.float64)
    n = int(values.numel())
    mean = float(values.mean().item())
    if n < 2:
        return mean
    standard_error = float(values.std(unbiased=True).item() / math.sqrt(n))
    z = normal_quantile(1.0 - delta)
    # Small-sample inflation for a one-sided Student-t-like interval without
    # depending on scipy inside the notebook.
    inflation = math.sqrt(n / max(n - 2, 1)) if n > 2 else 2.0
    return mean - z * inflation * standard_error


@torch.no_grad()
def estimate_nwj_bound(
    model,
    step_count,
    batch_size,
    batches,
    negative_count,
    delta,
    chunk_size=EVAL_LOGIT_CHUNK_SIZE,
    clamp_abs=CONFIDENCE_LOGIT_ABS_CLAMP,
):
    was_training = model.training
    model.eval()
    if str(DEVICE).startswith("cuda"):
        torch.cuda.empty_cache()

    batch_nwj = []
    positive_weighted_sum = 0.0
    negative_weighted_sum = 0.0
    positive_samples = 0
    negative_samples = 0
    for _ in range(batches):
        x, pad_mask, endpoint, _ = make_trajectories(batch_size, step_counts=step_count)
        positive_logit, negative_logit = contrastive_logits(
            model,
            x,
            pad_mask,
            endpoint,
            negative_count,
            negative_chunk_size=chunk_size,
        )
        summary = nwj_summary_from_logits(positive_logit, negative_logit, clamp_abs)
        batch_nwj.append(summary["nwj_nats"])
        positive_weighted_sum += (
            summary["positive_mean_nats"] * summary["positive_samples"]
        )
        negative_weighted_sum += (
            summary["negative_penalty_nats"] * summary["negative_samples"]
        )
        positive_samples += summary["positive_samples"]
        negative_samples += summary["negative_samples"]

    if was_training:
        model.train()

    nwj_mean_nats = float(np.mean(batch_nwj))
    nwj_lcb_nats = batch_mean_lcb(batch_nwj, delta)
    positive_mean_nats = positive_weighted_sum / max(positive_samples, 1)
    negative_penalty_nats = negative_weighted_sum / max(negative_samples, 1)
    return dict(
        nwj_bits=nwj_mean_nats / math.log(2.0),
        nwj_lcb_bits=nwj_lcb_nats / math.log(2.0),
        positive_mean_bits=positive_mean_nats / math.log(2.0),
        negative_penalty_bits=negative_penalty_nats / math.log(2.0),
        positive_samples=int(positive_samples),
        negative_samples=int(negative_samples),
        audit_batches=int(batches),
        delta=float(delta),
        clamp_abs=float(clamp_abs),
    )


def create_discriminator_bundle():
    model = EndpointTrajectoryDiscriminator().to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_EPOCHS * TRAIN_BATCHES_PER_EPOCH,
        eta_min=0.1 * LEARNING_RATE,
    )
    return model, optimizer, scheduler


_parameter_count_model = EndpointTrajectoryDiscriminator().to(
    device=DEVICE, dtype=DTYPE
)
sum(p.numel() for p in _parameter_count_model.parameters())

## 3. Training

BCE warmup, then NWJ with ramped hard negatives. Checkpoints require sane validation NWJ, neg_penalty, and pos.


In [ ]:
def validation_checkpoint_ok(validation, epoch):
    if epoch < VAL_CHECKPOINT_MIN_EPOCH:
        return False
    if not math.isfinite(validation["nwj_bits"]):
        return False
    if validation["nwj_bits"] < VAL_CHECKPOINT_MIN_NWJ_BITS:
        return False
    if validation["negative_penalty_bits"] > VAL_CHECKPOINT_MAX_NEG_PENALTY_BITS:
        return False
    if validation["positive_mean_bits"] > VAL_CHECKPOINT_MAX_POS_BITS:
        return False
    return True


def train_discriminator_for_steps(step_count):
    model, optimizer, scheduler = create_discriminator_bundle()
    history = []
    best_val_nwj_bits = -float("inf")
    best_state_dict = None
    best_epoch = None
    stale_nwj_epochs = 0
    model.train()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        if epoch == BCE_WARMUP_EPOCHS + 1:
            for group in optimizer.param_groups:
                group["lr"] = NWJ_LEARNING_RATE
        losses = []
        skipped_batches = 0
        bces = []
        accuracies = []
        pos_logits = []
        neg_logits = []
        phase = "BCE warmup" if epoch <= BCE_WARMUP_EPOCHS else "NWJ maximize"
        hard_fraction = hard_negative_fraction_for_epoch(
            epoch,
            BCE_WARMUP_EPOCHS,
            HARD_NEGATIVE_RAMP_EPOCHS,
            HARD_NEGATIVE_FRACTION,
        )
        in_batch_count = (
            IN_BATCH_NEGATIVES
            if epoch > BCE_WARMUP_EPOCHS + IN_BATCH_WARMUP_EPOCHS
            else 0
        )

        for _ in range(TRAIN_BATCHES_PER_EPOCH):
            negative_total = NEGATIVES_PER_POSITIVE + in_batch_count
            x, pad_mask, endpoint, labels = make_discriminator_batch(
                TRAIN_BATCH_SIZE, step_count, negative_total
            )
            positive_logit, negative_logit = contrastive_logits(
                model,
                x,
                pad_mask,
                endpoint,
                NEGATIVES_PER_POSITIVE,
                negative_chunk_size=EVAL_LOGIT_CHUNK_SIZE,
                hard_negative_fraction=hard_fraction,
                in_batch_negatives=in_batch_count,
            )
            logits = torch.cat([positive_logit, negative_logit], dim=0)
            bce = F.binary_cross_entropy_with_logits(logits, labels)
            if epoch <= BCE_WARMUP_EPOCHS:
                loss = bce
            else:
                loss = (
                    discriminator_nwj_training_loss(positive_logit, negative_logit)
                    + NWJ_BCE_WEIGHT * bce
                )

            if not torch.isfinite(loss):
                skipped_batches += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                skipped_batches += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                prediction = logits > 0.0
                losses.append(float(loss.detach().cpu()))
                bces.append(float(bce.detach().cpu()))
                accuracies.append(
                    float((prediction == labels.bool()).float().mean().cpu())
                )
                pos_logits.append(positive_logit.detach())
                neg_logits.append(negative_logit.detach())

        if not pos_logits:
            print(f"all batches skipped at epoch {epoch}; stopping this model")
            break
        pos_logits = torch.cat(pos_logits)
        neg_logits = torch.cat(neg_logits)
        should_validate = (
            epoch == 1
            or epoch % VALIDATION_EVERY == 0
            or epoch == TRAIN_EPOCHS
            or epoch >= NWJ_EARLY_STOP_MIN_EPOCHS
        )
        validation = None
        if should_validate:
            val_negatives = VALIDATION_NEGATIVES_PER_POSITIVE
            if (
                epoch >= VAL_CHECKPOINT_MIN_EPOCH
                and epoch % VAL_AUDIT_NEGATIVES_EVERY == 0
            ):
                val_negatives = EVAL_NEGATIVES_PER_POSITIVE
            validation = estimate_nwj_bound(
                model,
                step_count,
                VALIDATION_BATCH_SIZE,
                VALIDATION_BATCHES_PER_POINT,
                val_negatives,
                0.5,
                clamp_abs=LOGIT_ABS_CLAMP,
                chunk_size=EVAL_LOGIT_CHUNK_SIZE,
            )
        row = dict(
            epoch=epoch,
            phase=phase,
            loss=float(np.mean(losses)),
            bce=float(np.mean(bces)),
            accuracy=float(np.mean(accuracies)),
            skipped_batches=skipped_batches,
            train_nwj_bits=discriminator_nwj_training_bits(pos_logits, neg_logits),
            val_nwj_bits=np.nan if validation is None else validation["nwj_bits"],
            pos_logit_bits=float(
                (bounded_logits(pos_logits).mean() / math.log(2.0)).cpu()
            ),
            neg_penalty_bits=float(
                (nwj_terms(pos_logits, neg_logits)[1].mean() / math.log(2.0)).cpu()
            ),
        )
        history.append(row)

        if validation is not None and validation_checkpoint_ok(validation, epoch):
            if row["val_nwj_bits"] > best_val_nwj_bits + NWJ_EARLY_STOP_MIN_DELTA_BITS:
                best_val_nwj_bits = row["val_nwj_bits"]
                best_epoch = epoch
                best_state_dict = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                stale_nwj_epochs = 0
            else:
                stale_nwj_epochs += 1
        elif epoch > BCE_WARMUP_EPOCHS and validation is not None:
            stale_nwj_epochs += 1

        should_stop = (
            epoch >= NWJ_EARLY_STOP_MIN_EPOCHS
            and stale_nwj_epochs >= NWJ_EARLY_STOP_PATIENCE
            and best_state_dict is not None
        )

        if (
            epoch == 1
            or epoch % PLOT_EVERY == 0
            or epoch == TRAIN_EPOCHS
            or should_stop
        ):
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["bce"] for r in history],
                    mode="lines",
                    name="BCE",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["train_nwj_bits"] for r in history],
                    mode="lines",
                    name="train NWJ",
                    yaxis="y2",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history if math.isfinite(r["val_nwj_bits"])],
                    y=[
                        r["val_nwj_bits"]
                        for r in history
                        if math.isfinite(r["val_nwj_bits"])
                    ],
                    mode="lines+markers",
                    name="validation NWJ",
                    yaxis="y2",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["accuracy"] for r in history],
                    mode="lines",
                    name="accuracy",
                    yaxis="y3",
                )
            )
            fig.update_layout(
                title=f"Training NWJ discriminator v2, steps={step_count}, epoch {epoch}/{TRAIN_EPOCHS}, {phase}",
                xaxis_title="epoch",
                yaxis_title="BCE",
                yaxis2=dict(title="NWJ, bits", overlaying="y", side="right"),
                yaxis3=dict(
                    title="accuracy",
                    overlaying="y",
                    side="right",
                    anchor="free",
                    position=0.97,
                    range=[0.0, 1.0],
                ),
                height=390,
                margin=dict(l=40, r=90, t=50, b=40),
            )
            display(fig)

        if should_stop:
            print(
                f"early stop at epoch {epoch}: train_NWJ={row['train_nwj_bits']:.4f} bits, "
                f"val_NWJ={row['val_nwj_bits']:.4f} bits, BCE={row['bce']:.5f}, "
                f"best_val_NWJ={best_val_nwj_bits:.4f} bits at epoch {best_epoch}, "
                f"skipped={row['skipped_batches']}"
            )
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return {
        "model": model,
        "history": history,
        "step_count": step_count,
        "best_epoch": best_epoch,
        "best_val_nwj_bits": best_val_nwj_bits,
    }


trained_classifiers = {}
for steps in EVAL_STEP_COUNTS:
    trained_classifiers[int(steps)] = train_discriminator_for_steps(int(steps))

for steps, result in trained_classifiers.items():
    last = result["history"][-1]
    print(
        f"steps={steps:2d}: BCE={last['bce']:.4f}, acc={last['accuracy']:.4f}, "
        f"train_NWJ={last['train_nwj_bits']:.4f} bits, "
        f"best_val_NWJ={result['best_val_nwj_bits']:.4f} bits "
        f"at epoch {result['best_epoch']}, skipped={last['skipped_batches']}"
    )

## 4. 30-bit audit evaluation

Same NWJ audit protocol and row format as `discriminator.ipynb` (v1 baseline).


In [ ]:
rows = []
row_delta = EVAL_TOTAL_CONFIDENCE_DELTA / (
    len(trained_classifiers) * len(EVAL_STEP_COUNTS)
)
print(
    f"Per-row failure probability <= {row_delta:.3g}; "
    f"P(any plotted audit NWJ LCB is above its batch-mean target) <= {EVAL_TOTAL_CONFIDENCE_DELTA:.3g}"
)
print(
    f"Audit critic clamp = +/-{CONFIDENCE_LOGIT_ABS_CLAMP:.1f} nats; target={TARGET_AUDIT_BITS:.1f} bits"
)

for trained_steps, result in trained_classifiers.items():
    model = result["model"]
    for eval_steps in EVAL_STEP_COUNTS:
        metrics = estimate_nwj_bound(
            model,
            int(eval_steps),
            EVAL_BATCH_SIZE,
            EVAL_BATCHES_PER_POINT,
            EVAL_NEGATIVES_PER_POSITIVE,
            row_delta,
            clamp_abs=CONFIDENCE_LOGIT_ABS_CLAMP,
        )
        rows.append(
            dict(
                trained_steps=int(trained_steps),
                eval_steps=int(eval_steps),
                distance_m=int(eval_steps) * STEP_METERS,
                matched=int(trained_steps) == int(eval_steps),
                **metrics,
            )
        )

for row in rows:
    print(
        f"trained={row['trained_steps']:2d}, eval={row['eval_steps']:2d}, "
        f"distance={row['distance_m']:6.1f} m, "
        f"audit_NWJ_LCB={row['nwj_lcb_bits']:.4f} bits, "
        f"NWJ={row['nwj_bits']:.4f} bits, "
        f"pos={row['positive_mean_bits']:.4f} bits, "
        f"neg_penalty={row['negative_penalty_bits']:.4f} bits"
    )

In [ ]:
fig = go.Figure()
for trained_steps in sorted({row["trained_steps"] for row in rows}):
    curve = sorted(
        [row for row in rows if row["trained_steps"] == trained_steps],
        key=lambda row: row["eval_steps"],
    )
    fig.add_trace(
        go.Scatter(
            x=[row["eval_steps"] for row in curve],
            y=[row["nwj_lcb_bits"] for row in curve],
            mode="lines+markers",
            name=f"trained steps={trained_steps}",
            customdata=[
                [row["distance_m"], row["matched"], row["nwj_bits"], row["delta"]]
                for row in curve
            ],
            hovertemplate=(
                "trained steps="
                + str(trained_steps)
                + "<br>eval steps=%{x}<br>distance=%{customdata[0]:.0f} m"
                + "<br>matched=%{customdata[1]}"
                + "<br>audit NWJ LCB=%{y:.4f} bits"
                + "<br>NWJ=%{customdata[2]:.4f} bits"
                + "<br>row failure prob<=%{customdata[3]:.3g}<extra></extra>"
            ),
        )
    )

matched_rows = sorted(
    [row for row in rows if row["matched"]], key=lambda row: row["eval_steps"]
)
fig.add_trace(
    go.Scatter(
        x=[row["eval_steps"] for row in matched_rows],
        y=[row["nwj_lcb_bits"] for row in matched_rows],
        mode="markers+lines",
        name="matched-step envelope",
        line=dict(color="black", width=4),
        marker=dict(color="black", size=9),
    )
)
fig.update_layout(
    title="30-bit audit NWJ LCB (discriminator v2)",
    xaxis=dict(title="evaluation steps", dtick=1),
    yaxis=dict(title="audit NWJ lower confidence bound, bits"),
    height=520,
    margin=dict(l=50, r=55, t=50, b=45),
)
fig.show()